# 02 — Feature Availability and Missingness Audit

## 1. Objective

This notebook characterizes predictor availability and missingness in
Health System A before feature engineering or predictive model fitting.

The primary prediction design is already frozen:

- **Landmark:** ICU hour 6
- **Prediction window:** ICU hours 6–18
- **Development system:** Health System A
- **External validation system:** Health System B

Only information available up to and including ICU hour 6 is examined.

The goals of this notebook are to:

1. quantify feature availability before the landmark;
2. characterize patient-level missingness and measurement frequency;
3. identify variables that are too sparse for reliable primary modeling;
4. distinguish static variables from longitudinal measurements; and
5. define a reproducible patient-level feature specification using
   System A only.

No predictive model is trained in this notebook.

## 2. Data Loading

Load System A patient records and reconstruct the frozen primary cohort
using the same eligibility rules established in the cohort feasibility
audit.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

A_DIR = ROOT / "data" / "raw" / "training" / "training_setA"

files_a = sorted(A_DIR.glob("*.psv"))

print("Project root: repository root")
print(f"System A files: {len(files_a):,}")

assert len(files_a) == 20_336

Project root: repository root
System A files: 20,336


## 3. Frozen Primary Cohort Reconstruction

The primary cohort uses ICU hour 6 as the landmark and ICU hour 18 as
the end of the prediction window.

Patients are eligible when:

- their observed record contains ICU hour 6;
- their sepsis onset is not left-truncated;
- they have not developed sepsis by the landmark; and
- their outcome during the 6–18 hour window can be determined.

The cohort definition is reproduced here only to identify the patients
whose pre-landmark feature availability will be audited.

In [2]:
LANDMARK = 6
HORIZON_END = 18

records = []

for path in files_a:
    df = pd.read_csv(path, sep="|")

    iculos = df["ICULOS"].to_numpy()
    labels = df["SepsisLabel"].to_numpy()

    min_iculos = int(iculos.min())
    max_iculos = int(iculos.max())

    positive_hours = iculos[labels == 1]

    if len(positive_hours) == 0:
        septic = False
        reconstructed_onset = np.nan
        onset_left_truncated = False

    else:
        septic = True
        first_positive = int(positive_hours.min())

        onset_left_truncated = bool(labels[0] == 1)

        reconstructed_onset = (
            np.nan
            if onset_left_truncated
            else first_positive + 6
        )

    records.append(
        {
            "patient_id": path.stem,
            "min_iculos": min_iculos,
            "max_iculos": max_iculos,
            "septic": septic,
            "reconstructed_onset": reconstructed_onset,
            "onset_left_truncated": onset_left_truncated,
        }
    )

cohort = pd.DataFrame(records)

cohort["observed_at_landmark"] = (
    (cohort["min_iculos"] <= LANDMARK)
    & (cohort["max_iculos"] >= LANDMARK)
)

cohort["prevalent_sepsis"] = (
    cohort["reconstructed_onset"].notna()
    & (cohort["reconstructed_onset"] <= LANDMARK)
)

cohort["incident_sepsis_in_window"] = (
    cohort["reconstructed_onset"].notna()
    & (cohort["reconstructed_onset"] > LANDMARK)
    & (cohort["reconstructed_onset"] <= HORIZON_END)
)

cohort["complete_followup_to_horizon"] = (
    cohort["max_iculos"] >= HORIZON_END
)

cohort["outcome_observable"] = (
    cohort["incident_sepsis_in_window"]
    | cohort["complete_followup_to_horizon"]
)

cohort["eligible"] = (
    cohort["observed_at_landmark"]
    & (~cohort["onset_left_truncated"])
    & (~cohort["prevalent_sepsis"])
    & cohort["outcome_observable"]
)

cohort["outcome"] = (
    cohort["incident_sepsis_in_window"].astype(int)
)

analysis_ids = set(
    cohort.loc[cohort["eligible"], "patient_id"]
)

print(f"Eligible patients: {len(analysis_ids):,}")
print(
    "Positive events:",
    int(cohort.loc[cohort["eligible"], "outcome"].sum()),
)

Eligible patients: 18,699
Positive events: 371


## 4. Variable Groups

The dataset contains longitudinal physiological and laboratory
measurements together with demographic and administrative variables.

`SepsisLabel` is an outcome variable and is never used as a predictor.

`ICULOS` defines observation time and is used for landmark restriction
rather than as a clinical measurement.

In [3]:
example = pd.read_csv(files_a[0], sep="|")

all_columns = example.columns.tolist()

excluded_columns = {
    "SepsisLabel",
    "ICULOS",
}

candidate_predictors = [
    col for col in all_columns
    if col not in excluded_columns
]

print(f"Total columns: {len(all_columns)}")
print(f"Candidate predictor columns: {len(candidate_predictors)}")

candidate_predictors

Total columns: 41
Candidate predictor columns: 39


['HR',
 'O2Sat',
 'Temp',
 'SBP',
 'MAP',
 'DBP',
 'Resp',
 'EtCO2',
 'BaseExcess',
 'HCO3',
 'FiO2',
 'pH',
 'PaCO2',
 'SaO2',
 'AST',
 'BUN',
 'Alkalinephos',
 'Calcium',
 'Chloride',
 'Creatinine',
 'Bilirubin_direct',
 'Glucose',
 'Lactate',
 'Magnesium',
 'Phosphate',
 'Potassium',
 'Bilirubin_total',
 'TroponinI',
 'Hct',
 'Hgb',
 'PTT',
 'WBC',
 'Fibrinogen',
 'Platelets',
 'Age',
 'Gender',
 'Unit1',
 'Unit2',
 'HospAdmTime']

In [4]:
vital_columns = [
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "DBP",
    "Resp",
    "EtCO2",
]

lab_columns = [
    "BaseExcess",
    "HCO3",
    "FiO2",
    "pH",
    "PaCO2",
    "SaO2",
    "AST",
    "BUN",
    "Alkalinephos",
    "Calcium",
    "Chloride",
    "Creatinine",
    "Bilirubin_direct",
    "Glucose",
    "Lactate",
    "Magnesium",
    "Phosphate",
    "Potassium",
    "Bilirubin_total",
    "TroponinI",
    "Hct",
    "Hgb",
    "PTT",
    "WBC",
    "Fibrinogen",
    "Platelets",
]

static_context_columns = [
    "Age",
    "Gender",
    "Unit1",
    "Unit2",
    "HospAdmTime",
]

grouped_columns = (
    vital_columns
    + lab_columns
    + static_context_columns
)

assert len(grouped_columns) == len(candidate_predictors)
assert set(grouped_columns) == set(candidate_predictors)

variable_groups = pd.Series(
    {
        "Vitals": len(vital_columns),
        "Laboratory variables": len(lab_columns),
        "Static / context variables": len(static_context_columns),
        "Total candidate predictors": len(grouped_columns),
    },
    name="n_variables",
)

variable_groups.to_frame()

,n_variables
Vitals,8
Laboratory variables,26
Static / context variables,5
Total candidate predictors,39


## 5. Pre-Landmark Feature Availability

Feature availability is evaluated only within each eligible patient's
observed history up to and including ICU hour 6.

For each candidate predictor, two quantities are distinguished:

- **patient-level availability:** the proportion of eligible patients
  with at least one observed value before or at the landmark;
- **measurement frequency:** the number of observed measurements within
  the available pre-landmark history.

These quantities are examined before defining any imputation or
patient-level aggregation strategy.

In [5]:
availability_records = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre_landmark = df.loc[
        df["ICULOS"] <= LANDMARK,
        candidate_predictors,
    ]

    record = {
        "patient_id": patient_id,
        "n_pre_landmark_rows": len(pre_landmark),
    }

    for col in candidate_predictors:
        n_observed = int(pre_landmark[col].notna().sum())

        record[f"{col}__n_observed"] = n_observed
        record[f"{col}__available"] = n_observed > 0

    availability_records.append(record)

availability = pd.DataFrame(availability_records)

print(f"Audited patients: {len(availability):,}")

assert len(availability) == len(analysis_ids)

availability.head()

Audited patients: 18,699


,patient_id,n_pre_landmark_rows,HR__n_observed,HR__available,O2Sat__n_observed,O2Sat__available,Temp__n_observed,Temp__available,SBP__n_observed,SBP__available,...,Age__n_observed,Age__available,Gender__n_observed,Gender__available,Unit1__n_observed,Unit1__available,Unit2__n_observed,Unit2__available,HospAdmTime__n_observed,HospAdmTime__available
0,p000001,6,5,True,5,True,0,False,3,True,...,6,True,6,True,0,False,0,False,6,True
1,p000002,6,5,True,5,True,2,True,5,True,...,6,True,6,True,6,True,6,True,6,True
2,p000003,6,5,True,5,True,1,True,5,True,...,6,True,6,True,6,True,6,True,6,True
3,p000004,6,5,True,5,True,1,True,5,True,...,6,True,6,True,6,True,6,True,6,True
4,p000005,5,5,True,5,True,2,True,5,True,...,5,True,5,True,5,True,5,True,5,True


In [6]:
availability["n_pre_landmark_rows"].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

count    18699.000000
mean         5.284079
std          1.196471
min          1.000000
1%           1.000000
5%           3.000000
10%          3.000000
25%          5.000000
50%          6.000000
75%          6.000000
90%          6.000000
95%          6.000000
99%          6.000000
max          6.000000
Name: n_pre_landmark_rows, dtype: float64

## 6. Patient-Level Feature Availability

Patient-level availability is defined as whether an eligible patient has
at least one observed value for a variable during the available clinical
history up to and including ICU hour 6.

For each predictor, this section reports:

- the number and proportion of patients with at least one observation;
- the corresponding patient-level missingness rate;
- the median number of observed measurements among patients for whom the
  variable is available.

This analysis is descriptive only. No variable is excluded at this stage.

In [7]:
feature_availability_records = []

for col in candidate_predictors:
    available_col = f"{col}__available"
    count_col = f"{col}__n_observed"

    n_available = int(availability[available_col].sum())
    availability_rate = n_available / len(availability)

    observed_counts = availability.loc[
        availability[available_col],
        count_col,
    ]

    feature_availability_records.append(
        {
            "feature": col,
            "n_available": n_available,
            "availability_rate": availability_rate,
            "missingness_rate": 1 - availability_rate,
            "median_measurements_if_available": (
                observed_counts.median()
                if len(observed_counts) > 0
                else np.nan
            ),
        }
    )

feature_availability = pd.DataFrame(
    feature_availability_records
)

feature_availability = feature_availability.sort_values(
    "availability_rate",
    ascending=False,
).reset_index(drop=True)

feature_availability

,feature,n_available,availability_rate,missingness_rate,median_measurements_if_available
0,HospAdmTime,18699,1.000000,0.000000,6.0
1,Age,18699,1.000000,0.000000,6.0
2,Gender,18699,1.000000,0.000000,6.0
3,HR,18668,0.998342,0.001658,5.0
4,MAP,18638,0.996738,0.003262,5.0
5,O2Sat,18578,0.993529,0.006471,5.0
6,Resp,18562,0.992673,0.007327,5.0
7,SBP,17780,0.950853,0.049147,5.0
8,Temp,16997,0.908979,0.091021,2.0
9,Hct,11550,0.617680,0.382320,1.0


In [8]:
feature_availability_display = (
    feature_availability.copy()
)

feature_availability_display["availability_pct"] = (
    100 * feature_availability_display["availability_rate"]
)

feature_availability_display["missingness_pct"] = (
    100 * feature_availability_display["missingness_rate"]
)

feature_availability_display[
    [
        "feature",
        "n_available",
        "availability_pct",
        "missingness_pct",
        "median_measurements_if_available",
    ]
].round(2)

,feature,n_available,availability_pct,missingness_pct,median_measurements_if_available
0,HospAdmTime,18699,100.00,0.00,6.0
1,Age,18699,100.00,0.00,6.0
2,Gender,18699,100.00,0.00,6.0
3,HR,18668,99.83,0.17,5.0
4,MAP,18638,99.67,0.33,5.0
5,O2Sat,18578,99.35,0.65,5.0
6,Resp,18562,99.27,0.73,5.0
7,SBP,17780,95.09,4.91,5.0
8,Temp,16997,90.90,9.10,2.0
9,Hct,11550,61.77,38.23,1.0


### Availability Categories and Variable Roles

Feature missingness is not interpreted identically across variable types.

For longitudinal physiological and laboratory variables, missingness may
reflect clinical measurement practices and can itself contain information.

For static and contextual variables, repeated hourly values do not
represent repeated independent measurements.

Variables are therefore summarized by both clinical role and
patient-level availability before any exclusion or imputation rule is
defined.

In [9]:
def assign_variable_group(feature):
    if feature in vital_columns:
        return "Vital"
    elif feature in lab_columns:
        return "Laboratory"
    elif feature in static_context_columns:
        return "Static / context"
    else:
        return "Other"


def assign_availability_category(rate):
    if rate >= 0.90:
        return "High (≥90%)"
    elif rate >= 0.50:
        return "Moderate (50–<90%)"
    elif rate >= 0.10:
        return "Low (10–<50%)"
    elif rate > 0:
        return "Very low (<10%)"
    else:
        return "Unavailable (0%)"


feature_audit = feature_availability.copy()

feature_audit["variable_group"] = (
    feature_audit["feature"].map(assign_variable_group)
)

feature_audit["availability_category"] = (
    feature_audit["availability_rate"].map(
        assign_availability_category
    )
)

feature_audit[
    [
        "feature",
        "variable_group",
        "availability_rate",
        "missingness_rate",
        "availability_category",
        "median_measurements_if_available",
    ]
].sort_values(
    ["variable_group", "availability_rate"],
    ascending=[True, False],
).reset_index(drop=True)

,feature,variable_group,availability_rate,missingness_rate,availability_category,median_measurements_if_available
0,Hct,Laboratory,0.617680,0.382320,Moderate (50–<90%),1.0
1,Glucose,Laboratory,0.592545,0.407455,Moderate (50–<90%),1.0
2,Potassium,Laboratory,0.558426,0.441574,Moderate (50–<90%),1.0
3,Hgb,Laboratory,0.528905,0.471095,Moderate (50–<90%),1.0
4,BUN,Laboratory,0.522541,0.477459,Moderate (50–<90%),1.0
5,Chloride,Laboratory,0.516926,0.483074,Moderate (50–<90%),1.0
6,HCO3,Laboratory,0.516070,0.483930,Moderate (50–<90%),1.0
7,FiO2,Laboratory,0.500562,0.499438,Moderate (50–<90%),2.0
8,pH,Laboratory,0.492754,0.507246,Low (10–<50%),2.0
9,BaseExcess,Laboratory,0.480988,0.519012,Low (10–<50%),2.0


In [10]:
pd.crosstab(
    feature_audit["variable_group"],
    feature_audit["availability_category"],
)

availability_category,High (≥90%),Low (10–<50%),Moderate (50–<90%),Unavailable (0%),Very low (<10%)
variable_group,,,,,
Laboratory,0,14,8,0,4
Static / context,3,0,2,0,0
Vital,6,0,1,1,0


## 7. Primary Candidate Feature Set

Primary modeling variables are selected using Health System A only.

Variables with patient-level availability below 10% are excluded from
the primary feature set because their observed values are available for
too few patients to support stable estimation and transportability
assessment.

This threshold is applied before examining Health System B.

Missingness itself may still contain clinical information. For retained
longitudinal variables, missingness and measurement frequency will be
represented explicitly during feature construction rather than treated
only as a nuisance to be imputed.

In [11]:
PRIMARY_AVAILABILITY_THRESHOLD = 0.10

primary_features = feature_audit.loc[
    feature_audit["availability_rate"]
    >= PRIMARY_AVAILABILITY_THRESHOLD,
    "feature",
].tolist()

excluded_sparse_features = feature_audit.loc[
    feature_audit["availability_rate"]
    < PRIMARY_AVAILABILITY_THRESHOLD,
    [
        "feature",
        "variable_group",
        "availability_rate",
        "missingness_rate",
    ],
].copy()

print(f"Primary candidate features: {len(primary_features)}")
print(f"Excluded sparse features: {len(excluded_sparse_features)}")

excluded_sparse_features.sort_values(
    "availability_rate",
    ascending=False,
)

Primary candidate features: 34
Excluded sparse features: 5


,feature,variable_group,availability_rate,missingness_rate
34,Bilirubin_total,Laboratory,0.099952,0.900048
35,Fibrinogen,Laboratory,0.060217,0.939783
36,TroponinI,Laboratory,0.010214,0.989786
37,Bilirubin_direct,Laboratory,0.007220,0.992780
38,EtCO2,Vital,0.000000,1.000000


In [12]:
primary_feature_summary = pd.DataFrame(
    {
        "feature": primary_features,
    }
)

primary_feature_summary["variable_group"] = (
    primary_feature_summary["feature"].map(
        assign_variable_group
    )
)

primary_feature_summary.groupby(
    "variable_group"
).size().to_frame("n_features")

,n_features
variable_group,
Laboratory,22
Static / context,5
Vital,7


## 8. Repeated-Measurement Support and Static-Variable Consistency

Before defining patient-level feature summaries, this section evaluates
whether longitudinal variables have sufficient repeated measurements to
support temporal summaries such as change or trend.

Static and contextual variables are also checked for within-patient
consistency during the pre-landmark window.

These checks are performed before specifying the final feature
construction strategy.

In [13]:
retained_longitudinal_features = [
    feature
    for feature in primary_features
    if feature in vital_columns or feature in lab_columns
]

repeated_measurement_records = []

for col in retained_longitudinal_features:
    count_col = f"{col}__n_observed"

    counts = availability[count_col]

    n_at_least_1 = int((counts >= 1).sum())
    n_at_least_2 = int((counts >= 2).sum())
    n_at_least_3 = int((counts >= 3).sum())

    repeated_measurement_records.append(
        {
            "feature": col,
            "variable_group": assign_variable_group(col),
            "n_with_1plus": n_at_least_1,
            "pct_with_1plus": 100 * n_at_least_1 / len(availability),
            "n_with_2plus": n_at_least_2,
            "pct_with_2plus": 100 * n_at_least_2 / len(availability),
            "n_with_3plus": n_at_least_3,
            "pct_with_3plus": 100 * n_at_least_3 / len(availability),
        }
    )

repeated_measurements = pd.DataFrame(
    repeated_measurement_records
)

repeated_measurements.sort_values(
    ["variable_group", "pct_with_2plus"],
    ascending=[True, False],
).reset_index(drop=True).round(2)

,feature,variable_group,n_with_1plus,pct_with_1plus,n_with_2plus,pct_with_2plus,n_with_3plus,pct_with_3plus
0,FiO2,Laboratory,9360,50.06,6525,34.89,2897,15.49
1,pH,Laboratory,9214,49.28,5212,27.87,2388,12.77
2,BaseExcess,Laboratory,8994,48.10,5093,27.24,2289,12.24
3,PaCO2,Laboratory,8715,46.61,4122,22.04,1495,8.00
4,Hct,Laboratory,11550,61.77,3484,18.63,704,3.76
5,Glucose,Laboratory,11080,59.25,3389,18.12,1243,6.65
6,Potassium,Laboratory,10442,55.84,3022,16.16,534,2.86
7,Hgb,Laboratory,9890,52.89,2432,13.01,381,2.04
8,SaO2,Laboratory,4337,23.19,1910,10.21,663,3.55
9,Chloride,Laboratory,9666,51.69,1828,9.78,166,0.89


In [14]:
static_consistency_records = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre_landmark = df.loc[
        df["ICULOS"] <= LANDMARK,
        static_context_columns,
    ]

    record = {"patient_id": patient_id}

    for col in static_context_columns:
        record[f"{col}__n_unique"] = (
            pre_landmark[col].dropna().nunique()
        )

    static_consistency_records.append(record)

static_consistency = pd.DataFrame(
    static_consistency_records
)

static_consistency_summary = pd.DataFrame(
    {
        "feature": static_context_columns,
        "patients_with_multiple_values": [
            int(
                (
                    static_consistency[f"{col}__n_unique"] > 1
                ).sum()
            )
            for col in static_context_columns
        ],
    }
)

static_consistency_summary

,feature,patients_with_multiple_values
0,Age,0
1,Gender,0
2,Unit1,0
3,Unit2,0
4,HospAdmTime,0


## 9. Primary Feature Construction Strategy

Feature construction is designed to preserve clinically meaningful
information while limiting dimensionality and avoiding unstable temporal
summaries.

The strategy is defined using Health System A only.

### Static and contextual variables

Static/context variables are constant within patients during the
pre-landmark window and are represented by a single patient-level value.

### Vital signs

Vital signs are measured repeatedly for a substantial fraction of
patients. Retained vital signs are summarized using:

- the most recent value before or at the landmark;
- the pre-landmark mean;
- the pre-landmark minimum;
- the pre-landmark maximum.

### Laboratory variables

Most laboratory variables have only one observed value for many
patients. Retained laboratory variables are therefore represented by
the most recent observed value before or at the landmark.

Temporal slopes are not included in the primary feature representation
because repeated laboratory measurements are unavailable for a large
fraction of patients.

### Missingness

For each retained longitudinal variable, a binary availability indicator
is included to distinguish an unobserved measurement from an observed
clinical value.

Measurement counts are retained for later distribution-shift analysis
but are not included in the primary predictive representation.

This separates the primary clinical prediction representation from
health-system measurement intensity, which will be examined explicitly
as a potential source of transportability failure.

In [15]:
primary_vitals = [
    feature
    for feature in primary_features
    if feature in vital_columns
]

primary_labs = [
    feature
    for feature in primary_features
    if feature in lab_columns
]

primary_static_context = [
    feature
    for feature in primary_features
    if feature in static_context_columns
]

feature_specification = []

for feature in primary_static_context:
    feature_specification.append(
        {
            "feature": feature,
            "variable_group": "Static / context",
            "value_summaries": "single value",
            "availability_indicator": False,
            "measurement_count_primary": False,
        }
    )

for feature in primary_vitals:
    feature_specification.append(
        {
            "feature": feature,
            "variable_group": "Vital",
            "value_summaries": "last, mean, min, max",
            "availability_indicator": True,
            "measurement_count_primary": False,
        }
    )

for feature in primary_labs:
    feature_specification.append(
        {
            "feature": feature,
            "variable_group": "Laboratory",
            "value_summaries": "last",
            "availability_indicator": True,
            "measurement_count_primary": False,
        }
    )

feature_specification = pd.DataFrame(
    feature_specification
)

feature_specification

,feature,variable_group,value_summaries,availability_indicator,measurement_count_primary
0,HospAdmTime,Static / context,single value,False,False
1,Age,Static / context,single value,False,False
2,Gender,Static / context,single value,False,False
3,Unit2,Static / context,single value,False,False
4,Unit1,Static / context,single value,False,False
5,HR,Vital,"last, mean, min, max",True,False
6,MAP,Vital,"last, mean, min, max",True,False
7,O2Sat,Vital,"last, mean, min, max",True,False
8,Resp,Vital,"last, mean, min, max",True,False
9,SBP,Vital,"last, mean, min, max",True,False


## 10. ICU Unit Encoding Audit

`Unit1` and `Unit2` are treated separately in the raw data but may
represent mutually exclusive categories of the same patient-level
context variable.

Before constructing the final feature matrix, their joint observation
pattern is audited to determine whether they should remain as two
predictors or be represented as a single categorical feature.

This decision is made using Health System A only.

In [16]:
unit_records = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre_landmark = df.loc[
        df["ICULOS"] <= LANDMARK,
        ["Unit1", "Unit2"],
    ]

    unit1_values = pre_landmark["Unit1"].dropna().unique()
    unit2_values = pre_landmark["Unit2"].dropna().unique()

    unit1 = (
        unit1_values[0]
        if len(unit1_values) > 0
        else np.nan
    )

    unit2 = (
        unit2_values[0]
        if len(unit2_values) > 0
        else np.nan
    )

    unit_records.append(
        {
            "patient_id": patient_id,
            "Unit1": unit1,
            "Unit2": unit2,
        }
    )

unit_audit = pd.DataFrame(unit_records)

unit_audit.head()

,patient_id,Unit1,Unit2
0,p000001,NaN,NaN
1,p000002,0.0,1.0
2,p000003,1.0,0.0
3,p000004,0.0,1.0
4,p000005,1.0,0.0


In [17]:
unit_pattern = (
    unit_audit
    .value_counts(
        ["Unit1", "Unit2"],
        dropna=False,
    )
    .rename("n_patients")
    .reset_index()
)

unit_pattern["pct_patients"] = (
    100 * unit_pattern["n_patients"] / len(unit_audit)
)

unit_pattern

,Unit1,Unit2,n_patients,pct_patients
0,NaN,NaN,8805,47.088080
1,0.0,1.0,5027,26.883791
2,1.0,0.0,4867,26.028130


In [18]:
unit_summary = pd.Series(
    {
        "patients": len(unit_audit),

        "both_missing": int(
            (
                unit_audit["Unit1"].isna()
                & unit_audit["Unit2"].isna()
            ).sum()
        ),

        "only_Unit1_observed": int(
            (
                unit_audit["Unit1"].notna()
                & unit_audit["Unit2"].isna()
            ).sum()
        ),

        "only_Unit2_observed": int(
            (
                unit_audit["Unit1"].isna()
                & unit_audit["Unit2"].notna()
            ).sum()
        ),

        "both_observed": int(
            (
                unit_audit["Unit1"].notna()
                & unit_audit["Unit2"].notna()
            ).sum()
        ),

        "both_observed_sum_not_1": int(
            (
                unit_audit["Unit1"].notna()
                & unit_audit["Unit2"].notna()
                & (
                    (unit_audit["Unit1"]
                     + unit_audit["Unit2"]) != 1
                )
            ).sum()
        ),
    },
    name="System A",
)

unit_summary.to_frame()

,System A
patients,18699
both_missing,8805
only_Unit1_observed,0
only_Unit2_observed,0
both_observed,9894
both_observed_sum_not_1,0


### ICU Unit Encoding Decision

The audit confirms that `Unit1` and `Unit2` are not independent
predictors.

Across all eligible System A patients:

- the two variables are either both observed or both missing;
- no patient has only one of the two variables observed;
- whenever both are observed, their values sum to one.

They therefore represent mutually exclusive indicators of a single
patient-level ICU unit variable.

For subsequent modeling, the raw pair is replaced by a three-level
categorical variable:

- `Unit1`
- `Unit2`
- `Unknown`

`Unknown` represents patients for whom both raw unit indicators are
missing.

This encoding decision is based on Health System A only and is frozen
before examining Health System B.

In [19]:
unit_audit["ICU_unit"] = np.select(
    [
        (
            unit_audit["Unit1"].eq(1)
            & unit_audit["Unit2"].eq(0)
        ),
        (
            unit_audit["Unit1"].eq(0)
            & unit_audit["Unit2"].eq(1)
        ),
    ],
    [
        "Unit1",
        "Unit2",
    ],
    default="Unknown",
)

unit_category_summary = (
    unit_audit["ICU_unit"]
    .value_counts()
    .rename("n_patients")
    .to_frame()
)

unit_category_summary["pct_patients"] = (
    100
    * unit_category_summary["n_patients"]
    / len(unit_audit)
)

unit_category_summary

,n_patients,pct_patients
ICU_unit,,
Unknown,8805,47.088080
Unit2,5027,26.883791
Unit1,4867,26.028130


## 11. Final Primary Feature Specification

The primary feature representation is finalized after the availability,
repeated-measurement, static-consistency, and ICU-unit encoding audits.

The representation contains:

- three directly represented static variables;
- one three-level ICU-unit categorical variable;
- seven retained vital signs summarized longitudinally;
- twenty-two retained laboratory variables represented by their most
  recent pre-landmark values;
- availability indicators for all retained longitudinal variables.

Measurement counts are preserved for later distribution-shift analyses
but are not included in the primary predictive representation.

No information from Health System B is used in this specification.

In [20]:
final_static_numeric = [
    "Age",
    "Gender",
    "HospAdmTime",
]

final_static_categorical = [
    "ICU_unit",
]

final_vitals = primary_vitals.copy()
final_labs = primary_labs.copy()

assert len(final_static_numeric) == 3
assert len(final_static_categorical) == 1
assert len(final_vitals) == 7
assert len(final_labs) == 22

final_primary_variables = (
    final_static_numeric
    + final_static_categorical
    + final_vitals
    + final_labs
)

print(
    f"Final conceptual predictor variables: "
    f"{len(final_primary_variables)}"
)

Final conceptual predictor variables: 33


In [21]:
final_feature_specification_records = []

for feature in final_static_numeric:
    final_feature_specification_records.append(
        {
            "feature": feature,
            "variable_group": "Static",
            "primary_representation": "single value",
            "availability_indicator": False,
            "measurement_count_primary": False,
        }
    )

final_feature_specification_records.append(
    {
        "feature": "ICU_unit",
        "variable_group": "Static / categorical",
        "primary_representation": (
            "3-level categorical; Unknown reference"
        ),
        "availability_indicator": False,
        "measurement_count_primary": False,
    }
)

for feature in final_vitals:
    final_feature_specification_records.append(
        {
            "feature": feature,
            "variable_group": "Vital",
            "primary_representation": (
                "last, mean, min, max"
            ),
            "availability_indicator": True,
            "measurement_count_primary": False,
        }
    )

for feature in final_labs:
    final_feature_specification_records.append(
        {
            "feature": feature,
            "variable_group": "Laboratory",
            "primary_representation": "last",
            "availability_indicator": True,
            "measurement_count_primary": False,
        }
    )

final_feature_specification = pd.DataFrame(
    final_feature_specification_records
)

final_feature_specification

,feature,variable_group,primary_representation,availability_indicator,measurement_count_primary
0,Age,Static,single value,False,False
1,Gender,Static,single value,False,False
2,HospAdmTime,Static,single value,False,False
3,ICU_unit,Static / categorical,3-level categorical; Unknown reference,False,False
4,HR,Vital,"last, mean, min, max",True,False
5,MAP,Vital,"last, mean, min, max",True,False
6,O2Sat,Vital,"last, mean, min, max",True,False
7,Resp,Vital,"last, mean, min, max",True,False
8,SBP,Vital,"last, mean, min, max",True,False
9,Temp,Vital,"last, mean, min, max",True,False


In [22]:
representation_dimensions = pd.Series(
    {
        "static_numeric": 3,
        "ICU_unit_dummy_variables": 2,
        "vital_summary_features": len(final_vitals) * 4,
        "laboratory_value_features": len(final_labs),
        "longitudinal_availability_indicators": (
            len(final_vitals) + len(final_labs)
        ),
    },
    name="n_columns",
)

representation_dimensions.loc["total"] = (
    representation_dimensions.sum()
)

representation_dimensions.to_frame()

,n_columns
static_numeric,3
ICU_unit_dummy_variables,2
vital_summary_features,28
laboratory_value_features,22
longitudinal_availability_indicators,29
total,84


## 12. System A Patient-Level Feature Matrix

Construct the frozen patient-level predictor representation for all
eligible Health System A patients.

Only observations available up to and including ICU hour 6 are used.

The resulting matrix contains:

- static patient/context features;
- two dummy variables for ICU unit, with `Unknown` as the reference;
- four summaries for each retained vital sign;
- the most recent value for each retained laboratory variable;
- binary availability indicators for all retained longitudinal variables.

Missing clinical values are intentionally preserved as `NaN`.

No imputation, scaling, feature selection, or model fitting is performed
in this notebook. These preprocessing steps will be estimated using
training data only during internal validation.

In [23]:
feature_rows = []

eligible_outcomes = (
    cohort.loc[
        cohort["eligible"],
        ["patient_id", "outcome"],
    ]
    .set_index("patient_id")["outcome"]
    .to_dict()
)

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    # Strict landmark restriction
    pre = (
        df.loc[df["ICULOS"] <= LANDMARK]
        .sort_values("ICULOS")
        .copy()
    )

    assert len(pre) > 0
    assert pre["ICULOS"].max() <= LANDMARK

    row = {
        "patient_id": patient_id,
        "outcome": eligible_outcomes[patient_id],
    }

    # ---------------------------------------------------------
    # Static numeric variables
    # ---------------------------------------------------------
    for col in final_static_numeric:
        values = pre[col].dropna()

        row[col] = (
            values.iloc[0]
            if len(values) > 0
            else np.nan
        )

    # ---------------------------------------------------------
    # ICU unit categorical encoding
    # Unknown is the reference category:
    # Unknown -> (0, 0)
    # Unit1   -> (1, 0)
    # Unit2   -> (0, 1)
    # ---------------------------------------------------------
    unit1_values = pre["Unit1"].dropna()
    unit2_values = pre["Unit2"].dropna()

    if len(unit1_values) == 0 and len(unit2_values) == 0:
        icu_unit = "Unknown"

    else:
        unit1 = unit1_values.iloc[0]
        unit2 = unit2_values.iloc[0]

        if unit1 == 1 and unit2 == 0:
            icu_unit = "Unit1"

        elif unit1 == 0 and unit2 == 1:
            icu_unit = "Unit2"

        else:
            raise ValueError(
                f"Unexpected Unit1/Unit2 encoding for {patient_id}: "
                f"{unit1}, {unit2}"
            )

    row["ICU_unit_Unit1"] = int(icu_unit == "Unit1")
    row["ICU_unit_Unit2"] = int(icu_unit == "Unit2")

    # ---------------------------------------------------------
    # Vital signs:
    # last, mean, minimum, maximum + availability indicator
    # ---------------------------------------------------------
    for col in final_vitals:
        values = pre[col].dropna()

        row[f"{col}__last"] = (
            values.iloc[-1]
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__mean"] = (
            values.mean()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__min"] = (
            values.min()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__max"] = (
            values.max()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__available"] = int(
            len(values) > 0
        )

    # ---------------------------------------------------------
    # Laboratory variables:
    # most recent value + availability indicator
    # ---------------------------------------------------------
    for col in final_labs:
        values = pre[col].dropna()

        row[f"{col}__last"] = (
            values.iloc[-1]
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__available"] = int(
            len(values) > 0
        )

    feature_rows.append(row)

system_a_features = pd.DataFrame(feature_rows)

system_a_features.head()

,patient_id,outcome,Age,Gender,HospAdmTime,ICU_unit_Unit1,ICU_unit_Unit2,HR__last,HR__mean,HR__min,...,Phosphate__last,Phosphate__available,SaO2__last,SaO2__available,Lactate__last,Lactate__available,AST__last,AST__available,Alkalinephos__last,Alkalinephos__available
0,p000001,0,83.14,0,-0.03,0,0,110.0,97.8,89.0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
1,p000002,0,75.91,0,-98.60,0,1,94.0,68.2,56.0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
2,p000003,0,45.82,0,-1195.71,1,0,84.0,88.6,84.0,...,2.4,1,NaN,0,NaN,0,NaN,0,NaN,0
3,p000004,0,65.71,0,-8.77,0,1,107.0,107.8,103.5,...,NaN,0,98.0,1,NaN,0,NaN,0,NaN,0
4,p000005,0,28.09,1,-0.05,1,0,71.0,76.4,71.0,...,2.8,1,NaN,0,NaN,0,16.0,1,65.0,1


In [24]:
identifier_columns = [
    "patient_id",
    "outcome",
]

model_feature_columns = [
    col
    for col in system_a_features.columns
    if col not in identifier_columns
]

integrity_summary = pd.Series(
    {
        "patients": len(system_a_features),
        "model_feature_columns": len(model_feature_columns),
        "duplicate_patient_ids": int(
            system_a_features["patient_id"].duplicated().sum()
        ),
        "positive_events": int(
            system_a_features["outcome"].sum()
        ),
        "missing_outcomes": int(
            system_a_features["outcome"].isna().sum()
        ),
        "all_missing_feature_columns": int(
            system_a_features[
                model_feature_columns
            ].isna().all().sum()
        ),
    },
    name="System A feature matrix",
)

integrity_summary.to_frame()

,System A feature matrix
patients,18699
model_feature_columns,84
duplicate_patient_ids,0
positive_events,371
missing_outcomes,0
all_missing_feature_columns,0


In [25]:
availability_mismatches = []

for col in final_vitals + final_labs:
    value_col = f"{col}__last"
    indicator_col = f"{col}__available"

    mismatch = (
        (
            system_a_features[value_col].notna()
        )
        !=
        (
            system_a_features[indicator_col] == 1
        )
    )

    availability_mismatches.append(
        {
            "feature": col,
            "mismatched_patients": int(
                mismatch.sum()
            ),
        }
    )

availability_integrity = pd.DataFrame(
    availability_mismatches
)

availability_integrity

,feature,mismatched_patients
0,HR,0
1,MAP,0
2,O2Sat,0
3,Resp,0
4,SBP,0
5,Temp,0
6,DBP,0
7,Hct,0
8,Glucose,0
9,Potassium,0


## 13. Initial Feature-Matrix Integrity and Range Audit

Before saving the frozen System A feature matrix, perform final
mechanical checks on the engineered representation.

The audit evaluates:

- constant or degenerate feature columns;
- exact duplicate engineered columns;
- internal consistency of vital-sign summaries;
- validity of ICU-unit dummy encoding; and
- the range of observed engineered values.

No observations or variables are removed based on these checks unless a
clear construction or data-integrity problem is identified.

In [26]:
feature_unique_counts = (
    system_a_features[model_feature_columns]
    .nunique(dropna=True)
    .sort_values()
)

constant_features = (
    feature_unique_counts[
        feature_unique_counts <= 1
    ]
    .rename("n_unique_nonmissing")
    .to_frame()
)

print(
    f"Constant / single-valued feature columns: "
    f"{len(constant_features)}"
)

constant_features

Constant / single-valued feature columns: 0


,n_unique_nonmissing


In [27]:
duplicate_feature_pairs = []

feature_data = system_a_features[
    model_feature_columns
]

for i, col1 in enumerate(model_feature_columns):
    for col2 in model_feature_columns[i + 1:]:
        if feature_data[col1].equals(
            feature_data[col2]
        ):
            duplicate_feature_pairs.append(
                {
                    "feature_1": col1,
                    "feature_2": col2,
                }
            )

duplicate_features = pd.DataFrame(
    duplicate_feature_pairs,
    columns=[
        "feature_1",
        "feature_2",
    ],
)

print(
    f"Exact duplicate feature pairs: "
    f"{len(duplicate_features)}"
)

duplicate_features

Exact duplicate feature pairs: 0


,feature_1,feature_2


In [28]:
vital_integrity_records = []

for col in final_vitals:
    last = system_a_features[f"{col}__last"]
    mean = system_a_features[f"{col}__mean"]
    minimum = system_a_features[f"{col}__min"]
    maximum = system_a_features[f"{col}__max"]

    available = (
        system_a_features[f"{col}__available"] == 1
    )

    violations = (
        available
        & (
            (minimum > maximum)
            | (mean < minimum)
            | (mean > maximum)
            | (last < minimum)
            | (last > maximum)
        )
    )

    vital_integrity_records.append(
        {
            "feature": col,
            "summary_logic_violations": int(
                violations.sum()
            ),
        }
    )

vital_summary_integrity = pd.DataFrame(
    vital_integrity_records
)

vital_summary_integrity

,feature,summary_logic_violations
0,HR,0
1,MAP,0
2,O2Sat,0
3,Resp,0
4,SBP,0
5,Temp,0
6,DBP,0


In [29]:
unit_dummy_summary = pd.Series(
    {
        "patients": len(system_a_features),

        "Unit1_dummy_positive": int(
            system_a_features[
                "ICU_unit_Unit1"
            ].sum()
        ),

        "Unit2_dummy_positive": int(
            system_a_features[
                "ICU_unit_Unit2"
            ].sum()
        ),

        "Unknown_reference": int(
            (
                (
                    system_a_features[
                        "ICU_unit_Unit1"
                    ] == 0
                )
                &
                (
                    system_a_features[
                        "ICU_unit_Unit2"
                    ] == 0
                )
            ).sum()
        ),

        "both_dummies_positive": int(
            (
                (
                    system_a_features[
                        "ICU_unit_Unit1"
                    ] == 1
                )
                &
                (
                    system_a_features[
                        "ICU_unit_Unit2"
                    ] == 1
                )
            ).sum()
        ),
    },
    name="System A",
)

unit_dummy_summary.to_frame()

,System A
patients,18699
Unit1_dummy_positive,4867
Unit2_dummy_positive,5027
Unknown_reference,8805
both_dummies_positive,0


In [30]:
numeric_range_records = []

for col in model_feature_columns:
    values = system_a_features[col].dropna()

    if len(values) == 0:
        continue

    numeric_range_records.append(
        {
            "feature": col,
            "n_nonmissing": len(values),
            "min": values.min(),
            "p01": values.quantile(0.01),
            "median": values.median(),
            "p99": values.quantile(0.99),
            "max": values.max(),
        }
    )

numeric_range_audit = pd.DataFrame(
    numeric_range_records
)

numeric_range_audit

,feature,n_nonmissing,min,p01,median,p99,max
0,Age,18699,18.11,20.9600,64.85,88.0102,89.00
1,Gender,18699,0.00,0.0000,1.00,1.0000,1.00
2,HospAdmTime,18699,-3710.66,-640.6104,-2.82,0.7304,23.99
3,ICU_unit_Unit1,18699,0.00,0.0000,0.00,1.0000,1.00
4,ICU_unit_Unit2,18699,0.00,0.0000,0.00,1.0000,1.00
...,...,...,...,...,...,...,...
79,Lactate__available,18699,0.00,0.0000,0.00,1.0000,1.00
80,AST__last,2047,6.00,10.0000,42.00,3605.6600,9840.00
81,AST__available,18699,0.00,0.0000,0.00,1.0000,1.00
82,Alkalinephos__last,2005,15.00,28.0000,81.00,582.9600,3833.00


### Focused Clinical-Value Range Review

Extreme values are reviewed separately from binary availability
indicators and ICU-unit dummy variables.

This review is descriptive only. Extreme observations are not clipped,
winsorized, or removed solely because they are rare. Any exclusion would
require evidence of a clear data or construction error.

In [31]:
clinical_value_columns = (
    final_static_numeric
    + [
        f"{col}__last"
        for col in final_vitals
    ]
    + [
        f"{col}__last"
        for col in final_labs
    ]
)

clinical_range_review = []

for col in clinical_value_columns:
    values = system_a_features[col].dropna()

    clinical_range_review.append(
        {
            "feature": col,
            "n_nonmissing": len(values),
            "min": values.min(),
            "p001": values.quantile(0.001),
            "p01": values.quantile(0.01),
            "median": values.median(),
            "p99": values.quantile(0.99),
            "p999": values.quantile(0.999),
            "max": values.max(),
        }
    )

clinical_range_review = pd.DataFrame(
    clinical_range_review
)

clinical_range_review

,feature,n_nonmissing,min,p001,p01,median,p99,p999,max
0,Age,18699,18.11,18.44000,20.9600,64.85,88.0102,88.92000,89.00
1,Gender,18699,0.00,0.00000,0.0000,1.00,1.0000,1.00000,1.00
2,HospAdmTime,18699,-3710.66,-1504.28628,-640.6104,-2.82,0.7304,4.27302,23.99
3,HR__last,18668,28.00,39.00000,49.0000,83.00,131.5000,149.00000,176.00
4,MAP__last,18638,20.00,35.00000,51.0000,77.00,119.5000,148.00000,295.00
5,O2Sat__last,18578,27.00,66.57700,89.0000,99.00,100.0000,100.00000,100.00
6,Resp__last,18562,2.00,5.00000,9.0000,17.00,34.0000,44.71950,61.00
7,SBP__last,17780,35.00,66.00000,81.0000,116.00,177.0000,200.22100,219.50
8,Temp__last,16997,20.90,33.56000,35.0000,36.72,38.7800,39.72000,41.64
9,DBP__last,10525,23.00,28.00000,37.0000,59.50,95.3800,127.00000,221.00


In [32]:
clinical_range_review[
    [
        "feature",
        "min",
        "p001",
        "p01",
        "median",
        "p99",
        "p999",
        "max",
    ]
].round(3)

,feature,min,p001,p01,median,p99,p999,max
0,Age,18.11,18.440,20.960,64.85,88.010,88.920,89.00
1,Gender,0.00,0.000,0.000,1.00,1.000,1.000,1.00
2,HospAdmTime,-3710.66,-1504.286,-640.610,-2.82,0.730,4.273,23.99
3,HR__last,28.00,39.000,49.000,83.00,131.500,149.000,176.00
4,MAP__last,20.00,35.000,51.000,77.00,119.500,148.000,295.00
5,O2Sat__last,27.00,66.577,89.000,99.00,100.000,100.000,100.00
6,Resp__last,2.00,5.000,9.000,17.00,34.000,44.720,61.00
7,SBP__last,35.00,66.000,81.000,116.00,177.000,200.221,219.50
8,Temp__last,20.90,33.560,35.000,36.72,38.780,39.720,41.64
9,DBP__last,23.00,28.000,37.000,59.50,95.380,127.000,221.00


### Targeted Review of Extreme Clinical Values

A small number of observed values lie near or beyond ranges that warrant
manual verification.

These observations are inspected in the original pre-landmark records
before any decision is made.

No value is removed, clipped, winsorized, or corrected solely because
it is extreme.

In [33]:
extreme_value_checks = {
    "FiO2": lambda x: x > 1,
    "Temp": lambda x: (x < 25) | (x > 45),
    "MAP": lambda x: (x < 10) | (x > 250),
    "DBP": lambda x: (x < 10) | (x > 200),
    "BaseExcess": lambda x: (x < -50) | (x > 50),
}

extreme_records = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre = df.loc[
        df["ICULOS"] <= LANDMARK
    ].copy()

    for feature, rule in extreme_value_checks.items():
        observed = pre.loc[
            pre[feature].notna(),
            ["ICULOS", feature],
        ]

        if len(observed) == 0:
            continue

        mask = rule(observed[feature])

        for _, record in observed.loc[mask].iterrows():
            extreme_records.append(
                {
                    "patient_id": patient_id,
                    "feature": feature,
                    "ICULOS": int(record["ICULOS"]),
                    "value": record[feature],
                }
            )

extreme_value_audit = pd.DataFrame(
    extreme_records
)

print(
    f"Flagged pre-landmark measurements: "
    f"{len(extreme_value_audit)}"
)

extreme_value_audit

Flagged pre-landmark measurements: 11


,patient_id,feature,ICULOS,value
0,p000064,DBP,3,287.0
1,p002421,Temp,6,20.9
2,p009742,MAP,4,266.0
3,p013461,BaseExcess,6,100.0
4,p016065,MAP,6,295.0
5,p016774,MAP,6,274.0
6,p016800,FiO2,2,10.0
7,p016881,DBP,4,298.0
8,p018171,MAP,4,260.0
9,p018554,MAP,4,297.0


In [34]:
if len(extreme_value_audit) > 0:
    extreme_value_summary = (
        extreme_value_audit
        .groupby("feature")
        .agg(
            flagged_measurements=("value", "size"),
            affected_patients=("patient_id", "nunique"),
            minimum_flagged=("value", "min"),
            maximum_flagged=("value", "max"),
        )
        .reset_index()
    )
else:
    extreme_value_summary = pd.DataFrame()

extreme_value_summary

,feature,flagged_measurements,affected_patients,minimum_flagged,maximum_flagged
0,BaseExcess,1,1,100.0,100.0
1,DBP,3,3,221.0,298.0
2,FiO2,1,1,10.0,10.0
3,MAP,5,5,260.0,297.0
4,Temp,1,1,20.9,20.9


### Local Context Around Flagged Extreme Measurements

Flagged extreme values are reviewed together with nearby ICU observations
to determine whether they appear to be isolated recording artifacts or
part of a consistent clinical trajectory.

This inspection is descriptive only. No values are modified at this
stage.

In [35]:
context_records = []

for _, flagged in extreme_value_audit.iterrows():
    patient_id = flagged["patient_id"]
    feature = flagged["feature"]
    flagged_hour = int(flagged["ICULOS"])

    path = A_DIR / f"{patient_id}.psv"
    df = pd.read_csv(path, sep="|")

    context_end = min(
        flagged_hour + 2,
        LANDMARK,
    )

    nearby = df.loc[
        df["ICULOS"].between(
            flagged_hour - 2,
            context_end,
        ),
        ["ICULOS", feature],
    ]

    for _, row in nearby.iterrows():
        context_records.append(
            {
                "patient_id": patient_id,
                "feature": feature,
                "flagged_hour": flagged_hour,
                "flagged_value": flagged["value"],
                "context_hour": int(row["ICULOS"]),
                "context_value": row[feature],
            }
        )

extreme_value_context = pd.DataFrame(
    context_records
)

assert (
    extreme_value_context["context_hour"]
    <= LANDMARK
).all()

extreme_value_context

,patient_id,feature,flagged_hour,flagged_value,context_hour,context_value
0,p000064,DBP,3,287.0,2,NaN
1,p000064,DBP,3,287.0,3,287.00
2,p000064,DBP,3,287.0,4,49.00
3,p000064,DBP,3,287.0,5,46.00
4,p002421,Temp,6,20.9,4,36.15
5,p002421,Temp,6,20.9,5,36.50
6,p002421,Temp,6,20.9,6,20.90
7,p009742,MAP,4,266.0,2,100.00
8,p009742,MAP,4,266.0,3,100.00
9,p009742,MAP,4,266.0,4,266.00


### Frozen Physiologic Plausibility Rules

Review of the flagged measurements showed that the extreme values were
isolated observations inconsistent with adjacent measurements, or, in
the case of `FiO2 = 10`, inconsistent with the fraction-based encoding
used by the surrounding dataset.

A small set of physiologic plausibility rules is therefore defined
before predictive model fitting.

Values outside these ranges are treated as unavailable (`NaN`) rather
than clipped, winsorized, or replaced with an assumed corrected value.

The rules are:

- `FiO2`: values greater than 1 are invalid;
- `Temp`: values below 25 or above 45 °C are invalid;
- `MAP`: values below 10 or above 250 mmHg are invalid;
- `DBP`: values below 10 or above 200 mmHg are invalid;
- `BaseExcess`: values below -50 or above 50 are invalid.

These rules were defined using Health System A only and are frozen before
examining Health System B.

In [36]:
PLAUSIBILITY_RULES = {
    "FiO2": lambda x: (x >= 0) & (x <= 1),
    "Temp": lambda x: (x >= 25) & (x <= 45),
    "MAP": lambda x: (x >= 10) & (x <= 250),
    "DBP": lambda x: (x >= 10) & (x <= 200),
    "BaseExcess": lambda x: (x >= -50) & (x <= 50),
}


def apply_plausibility_rules(df):
    cleaned = df.copy()

    for feature, valid_rule in PLAUSIBILITY_RULES.items():
        observed = cleaned[feature].notna()

        invalid = (
            observed
            & (~valid_rule(cleaned[feature]))
        )

        cleaned.loc[invalid, feature] = np.nan

    return cleaned

In [37]:
cleaning_records = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre = df.loc[
        df["ICULOS"] <= LANDMARK
    ].copy()

    cleaned = apply_plausibility_rules(pre)

    for feature in PLAUSIBILITY_RULES:
        n_removed = int(
            (
                pre[feature].notna()
                & cleaned[feature].isna()
            ).sum()
        )

        if n_removed > 0:
            cleaning_records.append(
                {
                    "patient_id": patient_id,
                    "feature": feature,
                    "measurements_set_to_missing": n_removed,
                }
            )

cleaning_audit = pd.DataFrame(
    cleaning_records
)

cleaning_summary = (
    cleaning_audit
    .groupby("feature")
    .agg(
        affected_patients=("patient_id", "nunique"),
        measurements_set_to_missing=(
            "measurements_set_to_missing",
            "sum",
        ),
    )
    .reset_index()
)

cleaning_summary

,feature,affected_patients,measurements_set_to_missing
0,BaseExcess,1,1
1,DBP,3,3
2,FiO2,1,1
3,MAP,5,5
4,Temp,1,1


## 14. Rebuild the Frozen Cleaned Feature Matrix

The primary System A feature matrix is rebuilt after applying the frozen
physiologic plausibility rules to raw pre-landmark measurements.

Cleaning is performed before patient-level aggregation.

All other cohort definitions, feature definitions, and encoding rules
remain unchanged.

In [38]:
system_a_features_uncleaned = system_a_features.copy()

cleaned_feature_rows = []

for path in files_a:
    patient_id = path.stem

    if patient_id not in analysis_ids:
        continue

    df = pd.read_csv(path, sep="|")

    pre = (
        df.loc[df["ICULOS"] <= LANDMARK]
        .sort_values("ICULOS")
        .copy()
    )

    # Apply frozen plausibility rules before aggregation.
    pre = apply_plausibility_rules(pre)

    assert len(pre) > 0
    assert pre["ICULOS"].max() <= LANDMARK

    row = {
        "patient_id": patient_id,
        "outcome": eligible_outcomes[patient_id],
    }

    # ---------------------------------------------------------
    # Static numeric variables
    # ---------------------------------------------------------
    for col in final_static_numeric:
        values = pre[col].dropna()

        row[col] = (
            values.iloc[0]
            if len(values) > 0
            else np.nan
        )

    # ---------------------------------------------------------
    # ICU unit encoding
    # ---------------------------------------------------------
    unit1_values = pre["Unit1"].dropna()
    unit2_values = pre["Unit2"].dropna()

    if len(unit1_values) == 0 and len(unit2_values) == 0:
        icu_unit = "Unknown"

    else:
        unit1 = unit1_values.iloc[0]
        unit2 = unit2_values.iloc[0]

        if unit1 == 1 and unit2 == 0:
            icu_unit = "Unit1"

        elif unit1 == 0 and unit2 == 1:
            icu_unit = "Unit2"

        else:
            raise ValueError(
                f"Unexpected Unit1/Unit2 encoding for {patient_id}: "
                f"{unit1}, {unit2}"
            )

    row["ICU_unit_Unit1"] = int(
        icu_unit == "Unit1"
    )

    row["ICU_unit_Unit2"] = int(
        icu_unit == "Unit2"
    )

    # ---------------------------------------------------------
    # Vital signs
    # ---------------------------------------------------------
    for col in final_vitals:
        values = pre[col].dropna()

        row[f"{col}__last"] = (
            values.iloc[-1]
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__mean"] = (
            values.mean()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__min"] = (
            values.min()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__max"] = (
            values.max()
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__available"] = int(
            len(values) > 0
        )

    # ---------------------------------------------------------
    # Laboratory variables
    # ---------------------------------------------------------
    for col in final_labs:
        values = pre[col].dropna()

        row[f"{col}__last"] = (
            values.iloc[-1]
            if len(values) > 0
            else np.nan
        )

        row[f"{col}__available"] = int(
            len(values) > 0
        )

    cleaned_feature_rows.append(row)

system_a_features = pd.DataFrame(
    cleaned_feature_rows
)

system_a_features.head()

,patient_id,outcome,Age,Gender,HospAdmTime,ICU_unit_Unit1,ICU_unit_Unit2,HR__last,HR__mean,HR__min,...,Phosphate__last,Phosphate__available,SaO2__last,SaO2__available,Lactate__last,Lactate__available,AST__last,AST__available,Alkalinephos__last,Alkalinephos__available
0,p000001,0,83.14,0,-0.03,0,0,110.0,97.8,89.0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
1,p000002,0,75.91,0,-98.60,0,1,94.0,68.2,56.0,...,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0
2,p000003,0,45.82,0,-1195.71,1,0,84.0,88.6,84.0,...,2.4,1,NaN,0,NaN,0,NaN,0,NaN,0
3,p000004,0,65.71,0,-8.77,0,1,107.0,107.8,103.5,...,NaN,0,98.0,1,NaN,0,NaN,0,NaN,0
4,p000005,0,28.09,1,-0.05,1,0,71.0,76.4,71.0,...,2.8,1,NaN,0,NaN,0,16.0,1,65.0,1


In [39]:
model_feature_columns = [
    col
    for col in system_a_features.columns
    if col not in ["patient_id", "outcome"]
]

cleaned_matrix_summary = pd.Series(
    {
        "patients": len(system_a_features),
        "model_feature_columns": len(model_feature_columns),
        "duplicate_patient_ids": int(
            system_a_features["patient_id"]
            .duplicated()
            .sum()
        ),
        "positive_events": int(
            system_a_features["outcome"].sum()
        ),
        "missing_outcomes": int(
            system_a_features["outcome"]
            .isna()
            .sum()
        ),
        "all_missing_feature_columns": int(
            system_a_features[
                model_feature_columns
            ]
            .isna()
            .all()
            .sum()
        ),
    },
    name="Cleaned System A feature matrix",
)

cleaned_matrix_summary.to_frame()

,Cleaned System A feature matrix
patients,18699
model_feature_columns,84
duplicate_patient_ids,0
positive_events,371
missing_outcomes,0
all_missing_feature_columns,0


In [40]:
uncleaned = (
    system_a_features_uncleaned
    .set_index("patient_id")
)

cleaned = (
    system_a_features
    .set_index("patient_id")
)

assert uncleaned.index.equals(cleaned.index)
assert uncleaned.columns.equals(cleaned.columns)

same_values = (
    uncleaned.eq(cleaned)
    |
    (
        uncleaned.isna()
        & cleaned.isna()
    )
)

changed_mask = ~same_values

# Outcome must never change during predictor cleaning.
assert (
    changed_mask["outcome"].sum()
    == 0
)

feature_change_mask = changed_mask.drop(
    columns="outcome"
)

affected_patients = int(
    feature_change_mask.any(axis=1).sum()
)

affected_engineered_cells = int(
    feature_change_mask.sum().sum()
)

print(
    f"Patients affected by cleaning: "
    f"{affected_patients}"
)

print(
    f"Engineered feature values changed: "
    f"{affected_engineered_cells}"
)

Patients affected by cleaning: 11
Engineered feature values changed: 25


### Final Post-Cleaning Integrity Check

The complete integrity audit is repeated after physiologic plausibility
cleaning because the cleaned matrix, rather than the initial matrix, is
the artifact used for downstream modeling.

All checks below refer to the final frozen System A feature matrix.

In [41]:
final_feature_columns = [
    col
    for col in system_a_features.columns
    if col not in ["patient_id", "outcome"]
]

# ---------------------------------------------------------
# Basic matrix integrity
# ---------------------------------------------------------
basic_checks = {
    "patients": len(system_a_features),
    "model_features": len(final_feature_columns),
    "duplicate_patient_ids": int(
        system_a_features["patient_id"]
        .duplicated()
        .sum()
    ),
    "positive_events": int(
        system_a_features["outcome"].sum()
    ),
    "missing_outcomes": int(
        system_a_features["outcome"]
        .isna()
        .sum()
    ),
    "all_missing_columns": int(
        system_a_features[
            final_feature_columns
        ].isna().all().sum()
    ),
}

# ---------------------------------------------------------
# Constant columns
# ---------------------------------------------------------
final_unique_counts = (
    system_a_features[
        final_feature_columns
    ]
    .nunique(dropna=True)
)

n_constant_columns = int(
    (final_unique_counts <= 1).sum()
)

# ---------------------------------------------------------
# Exact duplicate columns
# ---------------------------------------------------------
n_duplicate_pairs = 0

for i, col1 in enumerate(final_feature_columns):
    for col2 in final_feature_columns[i + 1:]:
        if system_a_features[col1].equals(
            system_a_features[col2]
        ):
            n_duplicate_pairs += 1

# ---------------------------------------------------------
# Availability indicator consistency
# ---------------------------------------------------------
availability_mismatch_total = 0

for col in final_vitals + final_labs:
    value_present = (
        system_a_features[
            f"{col}__last"
        ].notna()
    )

    indicator_present = (
        system_a_features[
            f"{col}__available"
        ] == 1
    )

    availability_mismatch_total += int(
        (
            value_present
            != indicator_present
        ).sum()
    )

# ---------------------------------------------------------
# Vital-summary consistency
# ---------------------------------------------------------
vital_logic_violations = 0

for col in final_vitals:
    last = system_a_features[
        f"{col}__last"
    ]
    mean = system_a_features[
        f"{col}__mean"
    ]
    minimum = system_a_features[
        f"{col}__min"
    ]
    maximum = system_a_features[
        f"{col}__max"
    ]

    available = (
        system_a_features[
            f"{col}__available"
        ] == 1
    )

    violations = (
        available
        & (
            (minimum > maximum)
            | (mean < minimum)
            | (mean > maximum)
            | (last < minimum)
            | (last > maximum)
        )
    )

    vital_logic_violations += int(
        violations.sum()
    )

# ---------------------------------------------------------
# ICU-unit dummy consistency
# ---------------------------------------------------------
both_unit_dummies_positive = int(
    (
        (
            system_a_features[
                "ICU_unit_Unit1"
            ] == 1
        )
        &
        (
            system_a_features[
                "ICU_unit_Unit2"
            ] == 1
        )
    ).sum()
)

final_integrity_summary = pd.Series(
    {
        **basic_checks,
        "constant_feature_columns":
            n_constant_columns,
        "exact_duplicate_feature_pairs":
            n_duplicate_pairs,
        "availability_indicator_mismatches":
            availability_mismatch_total,
        "vital_summary_logic_violations":
            vital_logic_violations,
        "both_unit_dummies_positive":
            both_unit_dummies_positive,
    },
    name="Final cleaned System A matrix",
)

final_integrity_summary.to_frame()

,Final cleaned System A matrix
patients,18699
model_features,84
duplicate_patient_ids,0
positive_events,371
missing_outcomes,0
all_missing_columns,0
constant_feature_columns,0
exact_duplicate_feature_pairs,0
availability_indicator_mismatches,0
vital_summary_logic_violations,0


## 15. Freeze and Save System A Artifacts

The cleaned System A patient-level feature matrix and final feature
specification are now frozen.

The matrix was constructed using only information available up to ICU
hour 6 and contains no imputation or scaling.

Subsequent preprocessing parameters will be estimated from training data
only during internal validation.

The frozen artifacts are saved for use in downstream modeling.

In [42]:
PROCESSED_DIR = ROOT / "data" / "processed"
METADATA_DIR = ROOT / "data" / "metadata"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SYSTEM_A_FEATURE_PATH = (
    PROCESSED_DIR
    / "system_a_features.csv"
)

FEATURE_SPEC_PATH = (
    METADATA_DIR
    / "feature_specification.csv"
)

system_a_features.to_csv(
    SYSTEM_A_FEATURE_PATH,
    index=False,
)

final_feature_specification.to_csv(
    FEATURE_SPEC_PATH,
    index=False,
)

print(
    f"Saved feature matrix to:\n"
    f"{SYSTEM_A_FEATURE_PATH.relative_to(ROOT)}"
)

print(
    f"\nSaved feature specification to:\n"
    f"{FEATURE_SPEC_PATH.relative_to(ROOT)}"
)

Saved feature matrix to:
data/processed/system_a_features.csv

Saved feature specification to:
data/metadata/feature_specification.csv


In [43]:
saved_features = pd.read_csv(
    SYSTEM_A_FEATURE_PATH
)

saved_specification = pd.read_csv(
    FEATURE_SPEC_PATH
)

save_check = pd.Series(
    {
        "saved_patients": len(saved_features),
        "saved_total_columns": len(
            saved_features.columns
        ),
        "saved_model_features": (
            len(saved_features.columns) - 2
        ),
        "saved_positive_events": int(
            saved_features["outcome"].sum()
        ),
        "saved_specification_rows": len(
            saved_specification
        ),
    },
    name="Frozen artifacts",
)

save_check.to_frame()

,Frozen artifacts
saved_patients,18699
saved_total_columns,86
saved_model_features,84
saved_positive_events,371
saved_specification_rows,33


## 16. Final Status

**System A feature construction is frozen.**

The final development matrix contains:

- 18,699 eligible patients;
- 371 incident sepsis events;
- 84 engineered predictor columns;
- 33 conceptual predictor variables.

All cohort, cleaning, availability, encoding, and feature-construction
decisions were made using Health System A only.

Health System B remains untouched.

No imputation, scaling, model fitting, or predictive-performance
evaluation has been performed in this notebook.